# Lesson 28: Class Imbalance
**Module 5: ML Classification**

Interactive notebook for handling imbalanced datasets.

## Learning Objectives
- Understand imbalance problems
- Apply class weights and SMOTE
- Choose appropriate metrics
- Build robust classifiers for rare events

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve
np.random.seed(42)

# Create imbalanced fraud dataset (5% fraud)
n = 2000
n_fraud = 100
X_normal = np.random.randn(n - n_fraud, 4)
X_fraud = np.random.randn(n_fraud, 4) + 2  # Slight shift
X = np.vstack([X_normal, X_fraud])
y = np.array([0] * (n - n_fraud) + [1] * n_fraud)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
print(f'Training class distribution: {np.bincount(y_train)}')
print(f'Test class distribution: {np.bincount(y_test)}')

## 1. The Problem: Baseline Model

In [ ]:
# Fit without handling imbalance
baseline = LogisticRegression()
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

print('Baseline (no imbalance handling):')
print(classification_report(y_test, pred_baseline))

## 2. Class Weights

In [ ]:
# Fit with balanced class weights
weighted = LogisticRegression(class_weight='balanced')
weighted.fit(X_train, y_train)
pred_weighted = weighted.predict(X_test)

print('With class_weight="balanced":')
print(classification_report(y_test, pred_weighted))

## 3. Precision-Recall Curve

In [ ]:
# Compare PR curves
fig, ax = plt.subplots(figsize=(10, 6))

for model, name in [(baseline, 'Baseline'), (weighted, 'Weighted')]:
    probs = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, probs)
    ax.plot(recall, precision, label=name)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()
plt.show()

## 4. Threshold Adjustment

In [ ]:
# Lower threshold to catch more fraud
probs = weighted.predict_proba(X_test)[:, 1]
for threshold in [0.5, 0.3, 0.2, 0.1]:
    pred = (probs >= threshold).astype(int)
    from sklearn.metrics import precision_score, recall_score
    p = precision_score(y_test, pred)
    r = recall_score(y_test, pred)
    print(f'Threshold={threshold}: Precision={p:.3f}, Recall={r:.3f}')

## Exercise: Apply SMOTE
Use SMOTE to oversample the minority class.

In [ ]:
# Exercise: Install imbalanced-learn and apply SMOTE
# !pip install imbalanced-learn
# from imblearn.over_sampling import SMOTE
# X_resampled, y_resampled = SMOTE().fit_resample(X_train, y_train)

# YOUR CODE HERE

---
## Solution
<details>
<summary>Click to reveal</summary>

```python
# Install if needed
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE

# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {np.bincount(y_train)}')
print(f'After SMOTE: {np.bincount(y_resampled)}')

# Fit on resampled data
smote_model = LogisticRegression()
smote_model.fit(X_resampled, y_resampled)
pred_smote = smote_model.predict(X_test)

print('\nWith SMOTE:')
print(classification_report(y_test, pred_smote))
```
</details>